# Chatbot de reservas basado en reglas — Nivel 0 de la progresión de agentes

---

**Autor:** Borja Mora Méndez
**Contacto:** [borja.mora.mendez@gmail.com](mailto:borja.mora.mendez@gmail.com) · [LinkedIn](https://www.linkedin.com/in/borjamoramendez/)
**Repositorio:** [Data Analytics Portfolio](https://github.com/BORJAMOME/Data-Analytics-Portfolio)
**Categoría:** IA & Big Data · Agentes IA · Nivel 0 (motor de reglas)

---

**Nivel 0 (sin IA):** sistema de reservas de un restaurante que funciona con un menú de
opciones y reglas de negocio explícitas (aforo máximo, fecha y hora válidas). No usa NLP,
ni similitud semántica, ni ningún modelo de lenguaje.

Es el punto de partida de la categoría **Agentes IA** a propósito. Servirá de referencia
para medir cuánto ayudan, en los siguientes niveles, la similitud semántica y después un
LLM de verdad a entender lo que pide el usuario.

In [1]:
import pandas as pd

## 1. Configuración inicial: aforo y registro de reservas

In [2]:
# Aforo máximo del restaurante
AFORO_MAXIMO = 50

# DataFrame en memoria con las reservas activas
reservas = pd.DataFrame(
    columns=[
        "Nombre",
        "Fecha",
        "Hora",
        "Personas"
    ]
)

## 2. Motor de reglas: funciones de negocio

In [3]:
def personas_reservadas(fecha, hora):
    if reservas.empty:
        return 0

    filtro = (
        (reservas["Fecha"] == fecha)
        & (reservas["Hora"] == hora)
    )

    return reservas.loc[filtro, "Personas"].sum()


def hacer_reserva():
    global reservas

    nombre = input("Ingrese su nombre: ")
    fecha = input("Fecha (AAAA-MM-DD): ")
    hora = input("Hora (HH:MM): ")
    personas = int(input("Número de personas: "))

    ocupadas = personas_reservadas(fecha, hora)

    if ocupadas + personas <= AFORO_MAXIMO:

        nueva = pd.DataFrame({
            "Nombre": [nombre],
            "Fecha": [fecha],
            "Hora": [hora],
            "Personas": [personas]
        })

        reservas = pd.concat(
            [reservas, nueva],
            ignore_index=True
        )

        print("Reserva realizada con éxito.")

    else:
        print("No hay capacidad disponible.")


def consultar_disponibilidad():
    fecha = input("Fecha (AAAA-MM-DD): ")
    hora = input("Hora (HH:MM): ")

    ocupadas = personas_reservadas(fecha, hora)
    libres = AFORO_MAXIMO - ocupadas

    print(
        f"Disponibilidad para {fecha} a las {hora}: "
        f"{libres} personas."
    )


def cancelar_reserva():
    global reservas

    nombre = input("Ingrese su nombre: ")

    if nombre in reservas["Nombre"].values:

        reservas = reservas[
            reservas["Nombre"] != nombre
        ]

        print("Reserva cancelada.")

    else:
        print("No se encontró ninguna reserva con ese nombre.")


def mostrar_reservas():
    if reservas.empty:
        print("No hay reservas.")

    else:
        print("\nReservas actuales:")
        print(reservas)

## 3. Bucle conversacional basado en menú de opciones

El "chatbot" no entiende lenguaje natural: el usuario elige un número y el programa
ejecuta la regla que toca.

In [4]:
def chatbot():

    print("=" * 50)
    print("Bienvenido al sistema de reservas del restaurante.")
    print("=" * 50)

    while True:

        print("\nOpciones:")
        print("1. Hacer una reserva")
        print("2. Consultar disponibilidad")
        print("3. Cancelar una reserva")
        print("4. Mostrar todas las reservas")
        print("5. Salir")

        opcion = input("Seleccione una opción (1-5): ")

        if opcion == "1":
            hacer_reserva()

        elif opcion == "2":
            consultar_disponibilidad()

        elif opcion == "3":
            cancelar_reserva()

        elif opcion == "4":
            mostrar_reservas()

        elif opcion == "5":
            print(
                "Gracias por usar el sistema de reservas. "
                "¡Hasta luego!"
            )
            break

        else:
            print(
                "Opción no válida. "
                "Por favor, intente de nuevo."
            )


# Descomentar para ejecutar en modo interactivo (requiere terminal con input()):
# chatbot()

## 4. Conclusión

> **El hallazgo:** para lo más básico de un sistema de reservas basta con reglas explícitas (aforo, fecha y hora), sin ningún modelo de lenguaje. El problema es que solo funciona si el usuario sigue el menú numérico al pie de la letra.

**Recomendaciones:**

1. **Usarlo como referencia, no como producto.** En cuanto el cliente pide algo a su manera ("quiero una mesa para el sábado" en vez de pulsar el 1), el sistema no sabe qué hacer. Esa es la razón para meter similitud semántica en el siguiente nivel.
2. **Medir lo que cuesta no tener NLP.** Todo lo que no está en el menú (cambiar una reserva, preguntar por la carta, quejarse) acaba en manos de una persona. Esa fricción es la cifra con la que habrá que justificar un LLM.
3. **Mantener las reglas de negocio explícitas aunque haya IA.** El control de aforo y la validación de fecha y hora seguirán siendo lógica fija en los siguientes niveles. Lo único que cambia es cómo se entiende lo que pide el usuario.